## Changelog

| Version | Date | Changes |
|---------|------|---------|
| v1 | 2026-07-20 | Initial bump chart — all visible topics, plain circle nodes |
| v2 | 2026-07-20 | Limit chart to top 30 topics; node labels use bold text with bounding box |
| v3 | 2026-07-20 | Vertical per-hour gridlines; dual top+bottom x-axis with UTC/GMT+8/GMT+7 labels; `TOP_N` configurable from config cell |

---

## Part A — Framework

Imports, logging configuration, and all helper functions.

Functions are written as plain importable callables — not notebook-only code — so they
can be moved to `scripts/capture_trends.py` for the GitHub Actions cron layer without
modification (local-first-now / automation-later seam).

**robots.txt (trends24.in):** The site blocks named AI-training crawlers (ClaudeBot,
GPTBot, Google-Extended, etc.) but permits the generic `User-agent: *` catch-all
(`Disallow:` empty = allow all). `Content-Signal: ai-train=no, use=reference` — reading
ranked names into a chart is reference use, not model training. A plain `requests`
script with an honest, non-spoofed User-Agent is permitted.

**Selectors note:** `fetch_page` + `parse_timeline_columns` use `.list-container` /
`h3` / `ol li a` selectors that match the `context.md` sketch and align with
HamidByte's public scraper targeting the same site structure. `parse_detail_stats_table`
uses selectors from the same scraper's worldwide-page implementation; they have *not*
been confirmed against `/indonesia/` specifically — treat as a strong hypothesis.

In [1]:
import re
import hashlib
import logging
from datetime import datetime, timezone
from pathlib import Path
import colorsys

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import ticker

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("kestrel-trends")

In [3]:
def normalize_trend_name(name: str) -> str:
    """Strip leading '#', collapse whitespace, lowercase — cross-hour/cross-source dedup key."""
    return re.sub(r"\s+", " ", name.lstrip("#")).strip().lower()


def parse_tweet_count(raw: str) -> int | None:
    """Parse human-readable count like '74M Tweets' → 74_000_000; '1.2K' → 1200.
    Returns None on empty input or unrecognised format."""
    if not raw:
        return None
    m = re.match(r"([\d.]+)\s*([KMBkmb]?)", raw.replace(",", ""))
    if not m:
        return None
    n, suffix = float(m.group(1)), m.group(2).upper()
    mult = {"": 1, "K": 1_000, "M": 1_000_000, "B": 1_000_000_000}.get(suffix)
    return int(n * mult) if mult is not None else None


def parse_duration(raw: str) -> tuple[str, float | None]:
    """Parse trending-duration string like '2 hours 30 minutes' → (label, 2.5).

    Returns (raw_label, total_hours). total_hours is None if no time pattern matched.
    Handles: days, hours/hr, minutes/min, seconds/sec (case-insensitive, plural-tolerant).
    """
    label = (raw or "").strip()
    if not label:
        return (label, None)
    total_hours = 0.0
    found = False
    for qty_str, unit in re.findall(
        r"(\d+(?:\.\d+)?)\s*(day|hour|hr|minute|min|second|sec)s?", label, re.I
    ):
        qty = float(qty_str)
        u = unit.lower()
        if u.startswith("day"):
            total_hours += qty * 24
        elif u.startswith("hour") or u == "hr":
            total_hours += qty
        elif u.startswith("min"):
            total_hours += qty / 60
        elif u.startswith("sec"):
            total_hours += qty / 3600
        found = True
    return (label, total_hours if found else None)


def shorten_hour_label(label: str) -> str:
    """Extract HH:MMZ from a JS Date toString string for chart axis display.

    Trends24 h3 text is a full JS Date string:
      'Sun Jul 19 2026 16:40:28 GMT+0000 (Coordinated Universal Time)' → '16:40Z'
    Falls back to first 12 chars if the pattern doesn't match.
    The raw label is preserved in the CSV; this is chart-display-only formatting.
    """
    m = re.search(r"(\d{1,2}:\d{2}):\d{2}\s*GMT", str(label))
    return (m.group(1) + "Z") if m else str(label)[:12]

def format_tz_label(raw_label: str) -> str:
    """Multi-line x-axis label showing UTC / GMT+8 / GMT+7 for a raw JS Date string.

    Extracts HH:MM UTC from the JS Date string, then computes +7 and +8 offsets.
    Returns a 3-line string, e.g.:
      16:40Z
      00:40+8
      23:40+7
    Falls back to shorten_hour_label output if the pattern does not match.
    """
    import re as _re
    m = _re.search(r"(\d{1,2}):(\d{2}):\d{2}\s*GMT", str(raw_label))
    if not m:
        return shorten_hour_label(raw_label)
    total_min = int(m.group(1)) * 60 + int(m.group(2))

    def _fmt(mins: int) -> str:
        return f"{(mins // 60) % 24:02d}:{mins % 60:02d}"

    return f"{_fmt(total_min)}Z\n{_fmt(total_min + 480)}+8\n{_fmt(total_min + 420)}+7"


In [4]:
def fetch_page(url: str, ua: str, timeout: int) -> BeautifulSoup:
    """GET url with honest User-Agent; raise on HTTP error; return parsed BeautifulSoup."""
    resp = requests.get(url, headers={"User-Agent": ua}, timeout=timeout)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def parse_timeline_columns(soup: BeautifulSoup, min_trends: int = 30) -> list[dict]:
    """Parse every visible hour-column from trends24.in/indonesia/.

    Selectors (from context.md sketch + HamidByte scraper targeting trends24.in):
      column wrapper : .list-container
      hour label     : h3 (first h3 inside .list-container)
      trend items    : ol li a  (text = display name; title attr = tweet-count label)

    Grabs ALL visible columns in one request — free backfill redundancy (see context.md §4):
    a single missed hourly cron run is recovered by the next successful run as long as the
    site's own retention window covers the gap.

    Raises RuntimeError on schema drift (no columns found, or largest column has fewer
    than min_trends entries). Fail loud — do not silently write partial data.

    Returns list of column dicts:
      {hour_label: str|None,
       trends: [{rank, name, name_norm, tweet_count_raw, tweet_count}]}
    """
    columns = []
    for col_el in soup.select(".list-container"):
        label_el = col_el.select_one("h3")
        hour_label = label_el.get_text(strip=True) if label_el else None
        trends = []
        for rank, a_tag in enumerate(col_el.select("ol li a"), start=1):
            name = a_tag.get_text(strip=True)
            if not name:
                continue
            count_raw = a_tag.get("title", "") or ""
            trends.append({
                "rank": rank,
                "name": name,
                "name_norm": normalize_trend_name(name),
                "tweet_count_raw": count_raw if count_raw else None,
                "tweet_count": parse_tweet_count(count_raw),
            })
        if trends:
            columns.append({"hour_label": hour_label, "trends": trends})

    if not columns:
        raise RuntimeError(
            "Schema drift on trends24.in/indonesia/: no '.list-container' columns found. "
            "Open the page in browser devtools and verify the column-wrapper selector — "
            "the site may have changed its markup."
        )
    max_len = max(len(c["trends"]) for c in columns)
    if max_len < min_trends:
        raise RuntimeError(
            f"Schema drift: largest column has only {max_len} trends, "
            f"expected >= {min_trends}. Verify selectors in devtools."
        )
    return columns

In [5]:
def _safe_int(s) -> int | None:
    """Convert string to int, returning None on any type or value error."""
    try:
        return int(str(s).strip())
    except (TypeError, ValueError):
        return None


def parse_detail_stats_table(soup: BeautifulSoup) -> list[dict] | None:
    """Attempt to parse the Table-view detail stats from the same trends24.in page.

    Hypothesis from HamidByte/Twitter-Trending-Hashtags-Scraper-Python (worldwide page):
    the table is pre-rendered in the raw HTML under a CSS-toggled tab, not JS-injected —
    so plain requests+BeautifulSoup is sufficient if the markup is present.

    Selectors tried (in order):
      primary  : section#table .table-container-4 table.the-table tbody.list
      fallback : table.the-table tbody

    Expected column shape per <tr>:
      td.rank                   → current rank in the table listing
      td.topic > a.trend-link   → topic display name (and link)
      td.position               → best position ever reached
      td.count[data-count]      → total tweet count (prefer data-count attr, fall back to text)
      td.duration               → how long the topic has been trending (raw text)

    IMPORTANT: These selectors were confirmed on the worldwide trends24.in root page by
    HamidByte's scraper; they have NOT been confirmed against /indonesia/ specifically.
    Treat as a strong starting hypothesis. If this function returns None, check whether:
      (a) the /indonesia/ subpage simply does not have the table tab, or
      (b) the table is genuinely JS-injected → install Playwright and re-fetch with it.

    Returns list of detail dicts, or None if the table section is absent from raw HTML.
    When None, best_position/total_tweets/trending_for_* will be null in the output CSV.
    """
    tbody = soup.select_one(
        "section#table .table-container-4 table.the-table tbody.list"
    )
    if tbody is None:
        tbody = soup.select_one("table.the-table tbody")  # looser fallback
    if tbody is None:
        return None

    rows = []
    for tr in tbody.select("tr"):
        topic_a = tr.select_one("td.topic a.trend-link")
        if not topic_a:
            continue
        rank_el = tr.select_one("td.rank")
        pos_el = tr.select_one("td.position")
        count_el = tr.select_one("td.count")
        dur_el = tr.select_one("td.duration")

        count_raw = ""
        if count_el:
            count_raw = count_el.get("data-count", "") or count_el.get_text(strip=True)

        dur_label, dur_hours = parse_duration(
            dur_el.get_text(strip=True) if dur_el else ""
        )
        rows.append({
            "rank_table": _safe_int(rank_el.get_text(strip=True) if rank_el else None),
            "name": topic_a.get_text(strip=True),
            "name_norm": normalize_trend_name(topic_a.get_text(strip=True)),
            "best_position": _safe_int(pos_el.get_text(strip=True) if pos_el else None),
            "total_tweets": parse_tweet_count(count_raw),
            "trending_for_raw": dur_label if dur_label else None,
            "trending_for_hours": dur_hours,
        })
    return rows if rows else None

In [6]:
def build_pumpkin_palette(n: int = 150) -> list[str]:
    """Build a preset palette of n warm-autumn hex colors anchored on Pumpkin (#FF7518).

    Grid layout: 6 hue × 5 saturation × 5 lightness = 150 total.

    Hue range [4°, 69°]: pumpkin base ≈ 24° (spread -20° / +45°), covering
    deep red-orange → orange → burnt-orange → amber in one cohesive warm family.
    Saturation range [70%, 100%], Lightness range [35%, 65%].

    Colors are generated once (in Part B) and kept stable across runs.
    Topic → color assignment is via get_topic_color(name_norm, palette) which hashes
    name_norm modulo palette length — same topic always gets the same color.

    colorsys note: hls_to_rgb(h, l, s) takes HLS order (not HSL).
    """
    assert n == 150, (
        f"Grid 6×5×5=150 exactly; got n={n}. "
        "Adjust n_h/n_s/n_l and the assert if you need a different size."
    )
    n_h, n_s, n_l = 6, 5, 5
    hue_min, hue_max = 4 / 360, 69 / 360   # [4°, 69°] as colorsys [0,1] fractions
    sat_min, sat_max = 0.70, 1.00
    lit_min, lit_max = 0.35, 0.65

    hues = [hue_min + i * (hue_max - hue_min) / (n_h - 1) for i in range(n_h)]
    sats = [sat_min + i * (sat_max - sat_min) / (n_s - 1) for i in range(n_s)]
    lits = [lit_min + i * (lit_max - lit_min) / (n_l - 1) for i in range(n_l)]

    palette: list[str] = []
    for h in hues:
        for s in sats:
            for l in lits:
                # colorsys.hls_to_rgb takes (hue, lightness, saturation)
                r, g, b = colorsys.hls_to_rgb(h, l, s)
                r_i = min(255, int(r * 255))
                g_i = min(255, int(g * 255))
                b_i = min(255, int(b * 255))
                palette.append(f"#{r_i:02x}{g_i:02x}{b_i:02x}")
    return palette


def get_topic_color(name_norm: str, palette: list[str]) -> str:
    """Return a stable color for name_norm: MD5(name_norm) mod len(palette).

    Same name_norm always maps to the same color regardless of run order or
    how many other topics are present — palette reshuffling across runs is avoided.
    """
    idx = int(hashlib.md5(name_norm.encode()).hexdigest(), 16) % len(palette)
    return palette[idx]

In [7]:
def join_data(
    timeline_columns: list[dict],
    detail_stats: list[dict] | None,
    captured_at_utc: datetime,
    run_id: str,
    source_url: str,
) -> pd.DataFrame:
    """Flatten timeline columns and left-join detail stats on name_norm.

    Topics in the timeline that have no matching detail-stats row get None for
    best_position / total_tweets / trending_for_* — they are NOT dropped.
    Topics in detail_stats with no matching timeline entry are simply absent
    from the output (right-side-only rows are not included).

    Output schema (one row per topic per hour-column):
      captured_at_utc, run_id, source_url, hour_label, rank,
      name, name_norm, tweet_count_raw, tweet_count,
      best_position, total_tweets, trending_for_raw, trending_for_hours
    """
    detail_by_norm: dict[str, dict] = {}
    for row in (detail_stats or []):
        norm = row.get("name_norm", "")
        if norm:
            detail_by_norm[norm] = row

    rows = []
    for col in timeline_columns:
        for trend in col["trends"]:
            norm = trend["name_norm"]
            d = detail_by_norm.get(norm, {})
            rows.append({
                "captured_at_utc": captured_at_utc.isoformat(),
                "run_id": run_id,
                "source_url": source_url,
                "hour_label": col["hour_label"],
                "rank": trend["rank"],
                "name": trend["name"],
                "name_norm": norm,
                "tweet_count_raw": trend.get("tweet_count_raw"),
                "tweet_count": trend.get("tweet_count"),
                "best_position": d.get("best_position"),
                "total_tweets": d.get("total_tweets"),
                "trending_for_raw": d.get("trending_for_raw"),
                "trending_for_hours": d.get("trending_for_hours"),
            })
    return pd.DataFrame(rows)


def write_raw_csv(df: pd.DataFrame, path: Path) -> None:
    """Write flat denormalized DataFrame to CSV (UTF-8, no index)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8")
    log.info("CSV written: %s (%d rows)", path, len(df))

In [ ]:
def render_bump_chart(
    df: pd.DataFrame,
    palette: list[str],
    out_path: Path,
    top_n: int = 30,
) -> None:
    """Render and save a rank-over-time bump chart from the flat trends DataFrame.

    Layout:
      Y-axis : rank, inverted (rank 1 at the top), capped at top_n
      X-axis : hour_label columns (oldest left, newest right)
               Labels appear on BOTH top and bottom of the chart, each showing
               three timezone representations: UTC / GMT+8 / GMT+7
      Lines  : one per unique topic; NaN hours rendered as gaps (not interpolated)
      Color  : stable per topic via get_topic_color (Pumpkin palette, hash-assigned)
      Labels : topic name annotated at its rightmost non-NaN hour position,
               rendered as a bold boxed label for legibility
      Grid   : horizontal rank gridlines + vertical per-hour-column gridlines

    Only the top_n topics (by best rank achieved across all hours) are plotted.
    Topics beyond palette size (overflow) get dashed lines.
    """
    if df.empty:
        log.warning("DataFrame is empty — skipping bump chart render")
        return

    pivot = df.pivot_table(
        index="name_norm", columns="hour_label", values="rank", aggfunc="min"
    )
    pivot = pivot[pivot.columns[::-1]]
    pivot = pivot.loc[pivot.min(axis=1).sort_values().index]
    pivot = pivot.head(top_n)

    n_topics = len(pivot)
    n_hours = len(pivot.columns)
    palette_size = len(palette)
    overflow_count = max(0, n_topics - palette_size)

    fig_w = max(14, n_hours * 1.8)
    fig_h = max(10, min(n_topics * 0.35 + 4, 44))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=100)

    x_pos = list(range(n_hours))
    x_labels = [format_tz_label(lbl) for lbl in pivot.columns]

    for plot_idx, name_norm in enumerate(pivot.index):
        color = get_topic_color(name_norm, palette)
        linestyle = "--" if plot_idx >= palette_size else "-"
        y_vals = pivot.loc[name_norm].values

        ax.plot(
            x_pos, y_vals,
            color=color, linestyle=linestyle,
            linewidth=1.1, alpha=0.72,
            marker="o", markersize=3.5,
        )

        for xi in range(len(x_pos) - 1, -1, -1):
            if not pd.isna(y_vals[xi]):
                ax.annotate(
                    name_norm,
                    xy=(x_pos[xi], y_vals[xi]),
                    xytext=(6, 0), textcoords="offset points",
                    fontsize=6.0, va="center", color=color, clip_on=True,
                    fontweight="bold",
                    bbox=dict(
                        boxstyle="square,pad=0.25",
                        facecolor="white",
                        edgecolor=color,
                        linewidth=0.8,
                        alpha=0.88,
                    ),
                )
                break

    ax.invert_yaxis()
    ax.set_ylim(top_n + 0.5, 0.5)

    # Bottom x-axis: UTC / GMT+8 / GMT+7 timezone labels
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=90, ha="center", fontsize=6.5, linespacing=1.3)
    ax.tick_params(axis="x", which="major", bottom=True, labelbottom=True, top=False, labeltop=False)

    # Vertical gridlines (one per hour column) + horizontal rank gridlines
    ax.grid(True, axis="x", which="major", linestyle=":", alpha=0.28, color="#999999")
    ax.grid(True, axis="y", which="major", linestyle=":", alpha=0.35, color="#cccccc")

    ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(1))
    ax.set_ylabel("Rank (1 = top trend)", fontsize=10)
    ax.set_xlabel("Hour (left = oldest, right = most recent)", fontsize=9)
    ax.set_title(
        f"Indonesia Twitter/X Trends — Top {top_n} Bump Chart\n"
        f"Run: {out_path.parent.name}  |  {n_topics} topics shown  |  {n_hours} hour-columns",
        fontsize=11, pad=14,
    )

    # Top x-axis: mirror the same timezone labels above the chart
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(x_pos)
    ax_top.set_xticklabels(x_labels, rotation=90, ha="center", fontsize=6.5, linespacing=1.3)
    ax_top.tick_params(axis="x", which="major", length=4, pad=3)

    if overflow_count > 0:
        ax.annotate(
            f"\u26a0 {overflow_count} topic(s) exceed palette ({palette_size} colors); shown dashed",
            xy=(0.01, 0.01), xycoords="axes fraction", fontsize=6.5, color="#555555",
        )

    plt.tight_layout(pad=1.5)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    log.info(
        "Bump chart saved: %s (%d topics, %d hour-columns)", out_path, n_topics, n_hours
    )


## Part B — Config

Constants block and precomputed Pumpkin palette. All runtime-tunable values live here.
Nothing below should need changing for a normal run; adjust selectors in Part A if
markup has shifted.

In [9]:
# ── Target URL ────────────────────────────────────────────────────────────────
TRENDS24_URL = "https://trends24.in/indonesia/"

# ── Output root ───────────────────────────────────────────────────────────────
# Each run writes to dataset/trends/run_<YYYYMMDD-HHMMSS>/ so nothing overwrites
OUTPUT_ROOT = Path("dataset/trends")

# ── HTTP ──────────────────────────────────────────────────────────────────────
# Honest, non-spoofed UA — trends24.in robots.txt permits generic User-agent: * .
# Do not impersonate a browser UA; that would misrepresent the request.
USER_AGENT = (
    "Mozilla/5.0 (compatible; kestrel-trend-capture/1.0; "
    "+https://github.com/rednosereindeer80/kestrel-trending)"
)
REQUEST_TIMEOUT = 20  # seconds

# ── Schema-drift sanity guards ─────────────────────────────────────────────────
# parse_timeline_columns raises if the largest visible column has fewer trends than this.
# Verified range on trends24.in is ~50; 30 is a conservative floor (fail loud).
MIN_EXPECTED_TRENDS = 30

# ── Palette size ───────────────────────────────────────────────────────────────
# Must match build_pumpkin_palette()'s internal 6×5×5 grid.  Change both together.
PALETTE_SIZE = 150
# ── Chart display ──────────────────────────────────────────────────────────────
# Number of top-ranked topics to display in the bump chart.
# Supported values: 25, 30, 50.  Change only this constant; no other cell needs editing.
TOP_N = 30


In [10]:
PUMPKIN_PALETTE = build_pumpkin_palette(PALETTE_SIZE)
print(f"Palette: {len(PUMPKIN_PALETTE)} colors | hue 4°–69° | sat 70–100% | lit 35–65%")
print(f"Anchors : {PUMPKIN_PALETTE[0]}  {PUMPKIN_PALETTE[37]}  {PUMPKIN_PALETTE[74]}  {PUMPKIN_PALETTE[111]}  {PUMPKIN_PALETTE[149]}")

Palette: 150 colors | hue 4°–69° | sat 70–100% | lit 35–65%
Anchors : #97231a  #eb5013  #fea54c  #c8bc10  #e4fe4c


## Part C — Orchestration

One cell per step. Run top-to-bottom ("Run All") with no manual steps and no
required environment variables.

**Single-request design:** the page is fetched once in C2. Both the timeline parser
(C3) and detail-stats parser (C4) operate on the same `soup` object — only 1 HTTP
request total regardless of how many trends or hour-columns are visible.

In [11]:
# Compute run timestamp ONCE here, UTC. Reused for folder name, file names, run_id column.
# Never re-compute inside later cells — avoids folder/file stamp drift across slow cells.
RUN_TS = datetime.now(timezone.utc)
RUN_ID = RUN_TS.strftime("%Y%m%d-%H%M%S")
RUN_DIR = OUTPUT_ROOT / f"run_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = RUN_DIR / f"trends24_id_raw_{RUN_ID}.csv"
PNG_PATH = RUN_DIR / f"trends24_id_bumpchart_{RUN_ID}.png"

log.info("run_id=%s  output_dir=%s", RUN_ID, RUN_DIR)

02:09:13 [INFO] run_id=20260719-180913  output_dir=dataset\trends\run_20260719-180913


In [12]:
log.info("Fetching %s ...", TRENDS24_URL)
soup = fetch_page(TRENDS24_URL, USER_AGENT, REQUEST_TIMEOUT)
log.info("Fetch OK")

02:09:13 [INFO] Fetching https://trends24.in/indonesia/ ...
02:09:13 [INFO] Fetch OK


In [13]:
log.info("Parsing timeline columns ...")
timeline_columns = parse_timeline_columns(soup, min_trends=MIN_EXPECTED_TRENDS)

log.info(
    "Parsed %d hour-column(s): %s",
    len(timeline_columns),
    [c["hour_label"] for c in timeline_columns],
)
log.info(
    "Trends per column: %s",
    [len(c["trends"]) for c in timeline_columns],
)

02:09:13 [INFO] Parsing timeline columns ...
02:09:14 [INFO] Parsed 24 hour-column(s): ['Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 16:40:28 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 15:48:53 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 14:57:43 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 14:06:48 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 13:15:51 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 12:24:27 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 12:24:19 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 11:32:15 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 10:40:17 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 09:48:48 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 08:57:39 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 08:06:40 GMT+0000 (Coordinated Universal Time)', 'Sun Jul 19 2026 07:15:44 GMT+0000 (Coordinated Univer

In [14]:
log.info("Checking for detail-stats table in raw HTML ...")
detail_stats = parse_detail_stats_table(soup)

if detail_stats is None:
    log.warning(
        "Detail-stats table absent from raw HTML. "
        "Possible reasons: (a) the /indonesia/ subpage does not have the Table tab "
        "that the worldwide page has — check in a browser; "
        "(b) the table is JS-injected rather than pre-rendered — if so, install "
        "playwright (pip install playwright && playwright install chromium) and "
        "re-fetch with a headless browser. "
        "Proceeding with timeline data only; best_position / total_tweets / "
        "trending_for_* will be null in the output CSV."
    )
    detail_stats = []
else:
    log.info("Detail-stats table found in raw HTML: %d rows parsed", len(detail_stats))

02:09:14 [INFO] Checking for detail-stats table in raw HTML ...
02:09:14 [WARNING] Detail-stats table absent from raw HTML. Possible reasons: (a) the /indonesia/ subpage does not have the Table tab that the worldwide page has — check in a browser; (b) the table is JS-injected rather than pre-rendered — if so, install playwright (pip install playwright && playwright install chromium) and re-fetch with a headless browser. Proceeding with timeline data only; best_position / total_tweets / trending_for_* will be null in the output CSV.


In [15]:
log.info("Normalising + joining on name_norm ...")
df = join_data(timeline_columns, detail_stats, RUN_TS, RUN_ID, TRENDS24_URL)

total_rows = len(df)
unique_topics = df["name_norm"].nunique()
null_join_count = int(df["best_position"].isna().sum())

log.info(
    "DataFrame: %d rows | %d unique topics | %d rows missing detail-stats data",
    total_rows, unique_topics, null_join_count,
)
df.head(10)

02:09:14 [INFO] Normalising + joining on name_norm ...
02:09:14 [INFO] DataFrame: 1173 rows | 200 unique topics | 1173 rows missing detail-stats data


,captured_at_utc,run_id,source_url,hour_label,rank,name,name_norm,tweet_count_raw,tweet_count,best_position,total_tweets,trending_for_raw,trending_for_hours
0,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,1,#FINALBOLAGOPAY,finalbolagopay,None,None,None,None,None,None
1,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,2,Spain,spain,None,None,None,None,None,None
2,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,3,#PerthSantaConcertD2,perthsantaconcertd2,None,None,None,None,None,None
3,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,4,#BTSHALFTIMESHOW,btshalftimeshow,None,None,None,None,None,None
4,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,5,BTS AT WORLD CUP 2026,bts at world cup 2026,None,None,None,None,None,None
5,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,6,#HeartboundSeries,heartboundseries,None,None,None,None,None,None
6,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,7,Folaplay,folaplay,None,None,None,None,None,None
7,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,8,#EverSelcaDay,everselcaday,None,None,None,None,None,None
8,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,9,Trade $U,trade $u,None,None,None,None,None,None
9,2026-07-19T18:09:13.067530+00:00,20260719-180913,https://trends24.in/indonesia/,Sun Jul 19 2026 17:32:26 GMT+0000 (Coordinated...,10,RED VELVET COMEBACK SOON,red velvet comeback soon,None,None,None,None,None,None


In [16]:
write_raw_csv(df, CSV_PATH)

02:09:14 [INFO] CSV written: dataset\trends\run_20260719-180913\trends24_id_raw_20260719-180913.csv (1173 rows)


In [ ]:
log.info("Rendering bump chart ...")
render_bump_chart(df, PUMPKIN_PALETTE, PNG_PATH, top_n=TOP_N)


In [18]:
print("=" * 64)
print(f"Run ID         : {RUN_ID}")
print(f"Captured at    : {RUN_TS.isoformat()}")
print(f"Total rows     : {total_rows}")
print(f"Unique topics  : {unique_topics}")
print(f"Null joins     : {null_join_count} rows missing detail-stats fields")
print(f"Hour-columns   : {len(timeline_columns)}")
print(f"CSV            : {CSV_PATH}")
print(f"Bump chart     : {PNG_PATH}")
print("=" * 64)

Run ID         : 20260719-180913
Captured at    : 2026-07-19T18:09:13.067530+00:00
Total rows     : 1173
Unique topics  : 200
Null joins     : 1173 rows missing detail-stats fields
Hour-columns   : 24
CSV            : dataset\trends\run_20260719-180913\trends24_id_raw_20260719-180913.csv
Bump chart     : dataset\trends\run_20260719-180913\trends24_id_bumpchart_20260719-180913.png
